# zenz の追加学習（コンパウンドIME 用）

かな漢字変換の小さな人工知能 **zenz-v3.2-small** に、ウィキペディア・雑談・話し言葉・分野の文を、
**出典ごとの割合を決めて**追加で学ばせ、試験用の新しい模型（GGUF ファイル）を作ります。
数百万問の練習でも、何回かに分けて学習できます。

**使い方**

1. 上のメニューの「ランタイム」→「ランタイムのタイプを変更」で、**T4 GPU**（または L4）を選んで「保存」を押します
   （A100 でも動きます。設定は自動で合わせます）。
2. 下のセルを **上から順に1つずつ** 押します（セルの左の ▶ ボタン）。前のセルが終わってから次を押してください。
3. ③に、学習の時間・何回の接続に分かれるか・Google ドライブに置く大きさの見積りが出ます。
4. 途中で接続が切れたら、もう一度 **①から順に** 押してください。終わった所は飛ばし、練習問題は足りない部品だけ作り、
   学習は最後の保存（20分ごとにドライブへ）から続きます。数回に分けて学習するのが前提です。
5. **学習中はこのタブを開いたままにしてください。** 閉じると止まります。
6. 練習問題作り（CPU で数十分〜）に GPU の枠を使いたくないときは、①の `ONLY_DATA = True` にして、GPU なしのランタイムで
   ⑦まで先に済ませます。そのあと GPU のランタイムに替え、`ONLY_DATA = False` にして①から押し直します。

かかる時間の目安は③で出ます（T4、300万問・2周で、学習は合計 10 時間あまり。ほかに練習問題作りと後始末）。

## ① 設定
ふつうは変えなくてかまいません。▶ を押してください。

In [ ]:
#@title ① 設定
RUN_NAME = "v5-mix"        # 結果を置くフォルダの名前。別の設定で作り直すときは名前を変える
REPO_URL = "https://github.com/J-KAMMA/compound-ime.git"
REPO_REF = "exp/finetune-v5"   # 使うリポジトリの枝または版（main に入ったら "main" に）
DATA_SIZE = 3_000_000      # 学習用の問題の数（全部の出典の合計）
# 出典ごとの割合（合計は自動で 1 にそろえる）。分野は 10% に抑える（v3-domain は 50% で、ふつうの文が下がった）
#   wiki=ウィキペディア  persona=RealPersonaChat（雑談）  tatoeba=Tatoeba（話し言葉）
#   domain=プログラミング・仮想通貨の文書  synthetic=自作の一覧（話し言葉・略語・英語の言い回し）
# 材料の数の上限（2026-10-17 の試運転で実測。300万問のとき、3周まで使って作れる数）: domain 約26万、tatoeba 約7万、
# synthetic 約1.4万、persona は 66万以上、ウィキペディアは5分割で足りる。足りない分（約5.8万）はウィキペディアで埋まり、
# 実際の割合は wiki 66.4%・persona 22.0%・domain 8.8%・tatoeba 2.4%・synthetic 0.5% になる（⑦の最後にも出る）
# v5: 自作の一覧（synthetic）は、材料の数に合わせて 0.6% にした（v4 は 0.8% で、足りない分がウィキペディアで埋まっていた）
MIX = {"wiki": 0.645, "persona": 0.22, "tatoeba": 0.027, "domain": 0.10, "synthetic": 0.006}
# 出典の中の「問題の種類」の割合（v5 で足した。書かない種類は build_dataset.py の既定のまま）。
#  - wiki_whole を 0.54 に増やす: v4 は文全体の問題が全体の約 28% で、v1（32%）より減り、人名・地名の漢字が崩れた。
#    増やして全体の約 32%（v1 と同じ）に戻す。homophone は同音語の組を足したので 0.12 → 0.14。
#  - 自作の一覧: business（貴社・拝啓・お取引など）と counter（一杯・二羽・三日間など数＋助数詞）を足した。
#    材料の数（casual 約3,400、business 約1,000、counter 約4,100）に合わせて、足りない分が出ない割合にした
KIND_WEIGHTS = {"wiki_whole": 0.54, "homophone": 0.14,
                "casual": 0.24, "modern_words": 0.32, "english_phrases": 0.17, "business": 0.07, "counter": 0.20}
# 表記をそろえる文の割合。ウィキペディアと文の一覧（雑談・分野・Tatoeba）の文のうち、この割合の文で、
# 揺れのある語（すべて/全て、さまざま/様々、あす・あした/明日 など）を新聞・公用文の表記にそろえる
# （規則は tools/data/orthography.tsv、方針は README の「表記の方針」）。残りは元の表記のまま
ORTHO_SHARE = 0.8
# 出典ごとに「同じ文から作る問題の形の数」の上限。材料が足りないときだけ、この回数まで同じ文を別の形（文全体・
# 前の文つき・文の一部）で使い回す。それでも足りない分は、ウィキペディアの問題で埋める（manifest.json に記録）
MAX_PASSES = {"wiki": 2, "persona": 3, "tatoeba": 3, "domain": 3, "synthetic": 1}
EPOCHS = 2                 # 同じ問題を何回くり返すか
LEARNING_RATE = 3e-5       # 学習の歩幅（v1 の 3e-5 が、半分にした v2 より全体の試験で良かった）
EFFECTIVE_BATCH = 128      # 1歩で使う問題の数（GPU が違っても同じにする）
SEED = 20261015            # 乱数の種（同じなら同じ問題・同じ順番になる）
CHECKPOINT_MINUTES = 20    # 途中の結果をドライブに残す間隔（分）
KEEP_CHECKPOINTS = 1       # ドライブに残す途中の結果の数。1 個が約1.1GB。保存の最中だけ 2 個分になる
EVAL_PER_KIND = 300        # 成績を比べるときの、種類ごとの問題の数
RUN_MINUTES = 0            # 1回の実行で学習する時間の上限（分）。0 は制限なし（切れるまで）。数回に分けるなら 150 など。
                           # 上限で止まったら、⑨をもう一度押す（接続が切れたら①から押し直す）と、続きから始まる
DRIVE_BUDGET_GB = 4.0      # ドライブに置いてよい大きさの上限（GB）。見積りがこれを超えるなら、練習問題は Colab の手元に置く
DATA_ON_DRIVE = True       # True: 練習問題と数字にした問題をドライブに置く（切れても作り直さない）。③の見積りで足りなければ自動で False
ONLY_DATA = False          # True: 練習問題を作るところ（①〜⑦）だけ。GPU のない CPU のランタイムで先に作ると、GPU の枠を使わない
CLEAN_CHECKPOINTS = False  # True: 学習が終わったら途中の結果（約1.1GB）を消す。ドライブが足りないときに
BUILD_PROCESSES = 2        # 練習問題作りでウィキペディアを分けて並べる工程の数（Colab の CPU は 2 つ）。作りかけの途中で変えない

# ---- 固定した材料（版と SHA256。変えないでください）----
BASE_GGUF_URL = "https://huggingface.co/Miwa-Keita/zenz-v3.2-small-gguf/resolve/c67e03e07d215c869f591b274c1631170d3e11fe/ggml-model-Q5_K_M.gguf"
BASE_GGUF_SHA256 = "29c223d4c23327b80fd13ebb5ab2555057a46317997d5da391584ffbef0db673"
TOKENIZER_URL = "https://huggingface.co/Miwa-Keita/zenz-v3.1-small/resolve/344612e331bf3ba752a9bfef6633c93c6b933ebb/"
TOKENIZER_FILES = {
    "vocab.json": "67fd752abb091e649a9bb08bcb2b52b27cd6e3893ff5ee6e391e5555c0a90e0f",
    "merges.txt": "dde8cf4cc72f4d6eb7461a1c7dde6847916f0ad405395238aaf67501e2803ca7",
    "tokenizer_config.json": "bd7ac21f1da24ee416971702ffdd9c8489f0327914a067c65561f59c142f848d",
    "special_tokens_map.json": "f17f705b191ddb9aef0b42e40869d325d79b2a1e229e0d51d72698b5d3e4fc90",
}
# ウィキペディア日本語版（版 b04c8d1c…）。train-00003 と train-00011 は試験・選び手の材料に使っているので入れない。
# train-00007 は v1 から使っている。1つ約 220〜240MB（5つで約 1.1GB。Colab の手元に置く）
WIKI_URL = "https://huggingface.co/datasets/wikimedia/wikipedia/resolve/b04c8d1ceb2f5cd4588862100d08de323dccfbaa/"
WIKI_FILES = {
    "20231101.ja/train-00007-of-00015.parquet": "b112e83cf73327ba2246fd9c31eb1af501e51bac8450d8420963e29132d9b04b",
    "20231101.ja/train-00004-of-00015.parquet": "16bfb8cfc7b53084fb0e76ebaf9e3ab8e82d775a045e7781f96494a2277e2dae",
    "20231101.ja/train-00005-of-00015.parquet": "98fd014a7863168c5d23b8df3de874654bf1d858e5e67ffd003864ade5ae5723",
    "20231101.ja/train-00006-of-00015.parquet": "ea69e7a0ec68d5d3279a30820fb5bd496759daa32ec4f03146ff62a414ca5420",
    "20231101.ja/train-00008-of-00015.parquet": "859bb8856808e91d47cbf17f0c1ec5bfaf5ec8fb9f576c2f365bd49bda1daf7b",
}
# Tatoeba 日本語の文（CC BY 2.0 FR）。Tatoeba は毎週書き出しを入れ替え、古い版は消える（2026-10-10 の版 ba55cb50… も
# 2026-10-17 には取れなくなっていた）。そのため版は固定できない。初めて取った版をドライブの cache に控え、以後はそれを
# 使う。実際の SHA256 と日付は manifest.json に残る。固定したいときは、残った SHA256 をここに入れる（合わなければ止まる）
TATOEBA_URL = "https://downloads.tatoeba.org/exports/per_language/jpn/jpn_sentences.tsv.bz2"
TATOEBA_SHA256 = ""   # 空: 今の版（または cache の控え）を使う。試験の枠は文の番号で決まるので、版が違っても試験とは重ならない
MOZC_TABLE_URL = "https://raw.githubusercontent.com/unok/mozc/2d993fd532c0df849ff3a5f6c7851b3757471ed6/src/data/preedit/romanji-hiragana.tsv"
MOZC_TABLE_SHA256 = "fcb900e311d32ced05499c86e236d1084fb0c0937398deb6311fc6c7287a7821"
LLAMA_URL = "https://github.com/fkunn1326/llama.cpp.git"
LLAMA_COMMIT = "10131b23ee0dbd9d03dfc618cec9fe3402919277"   # コンパウンドIME が動かすときと同じ版

DRIVE_ROOT = "/content/drive/MyDrive/zenz-finetune"
RUN_DIR = f"{DRIVE_ROOT}/{RUN_NAME}"
WORK = "/content/work"
MIX_ARG = ",".join(f"{k}={v}" for k, v in MIX.items())
PASSES_ARG = ",".join(f"{k}={v}" for k, v in MAX_PASSES.items())
KIND_WEIGHTS_ARG = ",".join(f"{k}={v}" for k, v in KIND_WEIGHTS.items())
print("設定しました。結果の置き場所:", RUN_DIR)
print("出典の割合:", {k: f"{v / sum(MIX.values()):.1%}" for k, v in MIX.items()}, " 合計", f"{DATA_SIZE:,}", "問")

## ② GPU の確認と Google ドライブへの接続
▶ を押すと、Google ドライブへの接続の許可を求める画面が出ます。**自分のアカウントを選んで許可**してください。

In [ ]:
#@title ② GPU の確認と Google ドライブへの接続
import os, shutil, subprocess, sys, torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
elif ONLY_DATA:
    print("GPU なし（ONLY_DATA = True: 練習問題を作るところまでを、CPU のランタイムでやります）")
else:
    raise SystemExit("GPU がつながっていません。メニューの「ランタイム」→「ランタイムのタイプを変更」で "
                     "T4 GPU（または L4）を選んで保存し、①からやり直してください。"
                     "練習問題を作るだけなら、①の ONLY_DATA を True にすると GPU なしで進められます。")
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(WORK, exist_ok=True)
print("ドライブにつながりました:", RUN_DIR)
try:
    usage = shutil.disk_usage("/content/drive/MyDrive")
    print(f"ドライブの空き 約{usage.free / 1e9:.1f}GB（全体 約{usage.total / 1e9:.1f}GB）")
except OSError:
    pass
print(f"Colab の手元の空き 約{shutil.disk_usage(WORK).free / 1e9:.0f}GB、CPU {os.cpu_count()} 個、"
      f"メモリ {os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9:.1f}GB")

## ③ 道具（この学習の台本）を取ってきて、時間とユニットを見積もる

In [ ]:
#@title ③ 台本を取ってきて見積もる
import os, subprocess, sys, json
REPO = f"{WORK}/compound-ime"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin", REPO_REF], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", "FETCH_HEAD"], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
KIT = f"{REPO}/tools/zenz-finetune"
if not os.path.exists(f"{KIT}/zenz_tune.py"):
    # まだ GitHub に上がっていないときは、ドライブの zenz-finetune/kit フォルダに置いた台本を使う
    KIT = f"{DRIVE_ROOT}/kit"
    if not os.path.exists(f"{KIT}/zenz_tune.py"):
        raise SystemExit("台本が見つかりません。tools/zenz-finetune の中身（build_dataset.py と zenz_tune.py）を "
                         f"Google ドライブの {KIT} に置いてから、もう一度押してください。")
sys.path.insert(0, KIT)
import importlib, zenz_tune as z
importlib.reload(z)
print("リポジトリの版:", REPO_COMMIT, " 台本:", KIT)

PROFILE = z.gpu_profile(EFFECTIVE_BATCH)
if PROFILE["kind"] == "CPU":      # ONLY_DATA で GPU がないとき: 学習の見積りだけ T4 で出す
    PROFILE = dict(z.PROFILES["T4"], kind="T4", name="T4（見積り用。今は GPU なし）")
    PROFILE["micro"] = min(PROFILE["micro"], EFFECTIVE_BATCH)
    PROFILE["accum"] = max(1, EFFECTIVE_BATCH // PROFILE["micro"])

# ---- ドライブの容量の見積りと、練習問題の置き場所
plan = z.drive_plan(DATA_SIZE, keep=KEEP_CHECKPOINTS, data_on_drive=DATA_ON_DRIVE)
if DATA_ON_DRIVE and plan["peak_gb"] > DRIVE_BUDGET_GB:
    print(f"ドライブの見積りが {plan['peak_gb']:.1f}GB で、上限 {DRIVE_BUDGET_GB}GB を超えます。"
          "練習問題と数字にした問題は Colab の手元に置きます（ドライブには最小限）。")
    print("※ 手元の物は接続が切れると消えるので、そのときは⑦と⑧をやり直すことになります（問題作りに時間がかかります）。")
    DATA_ON_DRIVE = False
    plan = z.drive_plan(DATA_SIZE, keep=KEEP_CHECKPOINTS, data_on_drive=False)
DATA_DIR = f"{RUN_DIR}/data" if DATA_ON_DRIVE else f"{WORK}/data"
NPZ_DIR = RUN_DIR if DATA_ON_DRIVE else WORK
data_built = z.data_ready(DATA_DIR)

est = z.estimate(PROFILE, DATA_SIZE, EPOCHS, data_built=data_built, run_minutes=RUN_MINUTES or 170)
print(f"GPU の種類: {PROFILE['kind']}（{PROFILE['name']}）")
print(f"計算の精度: {PROFILE['precision']}、一度に {PROFILE['micro']} 問 × {PROFILE['accum']} 回で1歩")
print(f"見積り: 学習 約{est['train_minutes'] / 60:.1f} 時間（{DATA_SIZE:,} 問 × {EPOCHS} 回 × 平均 {z.AVG_TOKENS} 字 ÷ "
      f"{PROFILE['tok_s']:,} 字/秒）、練習問題作り 約{est['build_minutes']:.0f} 分（CPU）、全体 約{est['total_minutes'] / 60:.1f} 時間")
if RUN_MINUTES:
    print(f"1回の実行を {RUN_MINUTES} 分までにすると、学習は約 {est['runs']} 回に分かれます（切れても続きから）")
else:
    print(f"1回の接続で 約170 分学習できるとすると、学習は約 {est['runs']} 回に分かれます（切れたら①から押し直すと続きから）")
if est["units"] is not None:
    print(f"コンピューティング ユニット: 約{est['units']:.1f}（GPU をつなぐ時間だけなら 約{est['units_gpu_only']:.1f}。"
          f"1時間あたり {PROFILE['units_per_hour']}）")
print("\nGoogle ドライブに置くものの見積り（GB）:" + ("" if DATA_ON_DRIVE else "（練習問題は手元に置く場合）"))
for label, size in plan["rows"]:
    print(f"  {size:6.2f}  {label}")
print(f"  合計 約{plan['steady_gb']:.1f}GB、保存の最中は 約{plan['peak_gb']:.1f}GB（上限 {DRIVE_BUDGET_GB}GB）")
print("※ 学習の速さと容量の一部は仮の値です。学習が始まると実際の速さで残り時間を出し直します。")

## ④ 道具を入れる
版を固定した部品を入れます（2〜3分）。

In [ ]:
#@title ④ 道具を入れる
!pip -q install "transformers==4.57.6" "sudachipy==0.6.11" "sudachidict-core==20260428" "sentencepiece==0.2.1" "protobuf>=4.21,<7"
import transformers, sudachipy, torch
print("transformers", transformers.__version__, " torch", torch.__version__, " sudachipy", sudachipy.__version__)

## ⑤ 材料を取ってくる
元の模型、分かち方の部品、ローマ字表、llama.cpp を取ってきます。練習問題をまだ作っていないときは、ウィキペディアの
5つの分割（約1.1GB、Colab の手元）と Tatoeba の書き出しも取ります。ウィキペディアは SHA256（ファイルの指紋）を確かめます。
Tatoeba は古い版が消えるので固定できず、取った版をドライブに控えて実際の指紋を記録します（5〜10分）。

In [ ]:
#@title ⑤ 材料を取ってくる
import os, subprocess, urllib.request, datetime
BASE_GGUF = z.download(BASE_GGUF_URL, f"{DRIVE_ROOT}/cache/zenz-v3.2-small-Q5_K_M.gguf", BASE_GGUF_SHA256)
TOKENIZER_DIR = f"{DRIVE_ROOT}/cache/tokenizer"
for name, digest in TOKENIZER_FILES.items():
    z.download(TOKENIZER_URL + name, f"{TOKENIZER_DIR}/{name}", digest)
ROMAJI_TABLE = z.download(MOZC_TABLE_URL, f"{DRIVE_ROOT}/cache/romanji-hiragana.tsv", MOZC_TABLE_SHA256)

# ---- 練習問題の材料（まだ作っていない部品の分だけ取る）
WIKI_PROCS = BUILD_PROCESSES if MIX.get("wiki", 0) > 0 else 0
WIKI_PART_NAMES = [f"wiki{i}of{WIKI_PROCS}" for i in range(WIKI_PROCS)]
TEXT_SOURCES_USED = [s for s in ("synthetic", "domain", "tatoeba", "persona") if MIX.get(s, 0) > 0]
need_text = bool(TEXT_SOURCES_USED) and not z.part_done(DATA_DIR, "text")
need_wiki = any(not z.part_done(DATA_DIR, n) for n in WIKI_PART_NAMES)
WIKI_PATHS, TATOEBA_PATH, TATOEBA_INFO = [], None, {}
if z.data_ready(DATA_DIR):
    print("練習問題は作り済みなので、材料は取りません")
else:
    if need_wiki:
        for name, digest in WIKI_FILES.items():
            WIKI_PATHS.append(str(z.download(WIKI_URL + name, f"{WORK}/wiki/{os.path.basename(name)}", digest)))
        print("ウィキペディア:", [os.path.basename(p) for p in WIKI_PATHS])
    if need_text and MIX.get("tatoeba", 0) > 0:
        # 取った書き出しはドライブに控える（Tatoeba は古い版を消すので、次からはこの控えを使う）
        TATOEBA_PATH = str(z.download(TATOEBA_URL, f"{DRIVE_ROOT}/cache/jpn_sentences.tsv.bz2", TATOEBA_SHA256 or None))
        try:
            modified = urllib.request.urlopen(urllib.request.Request(TATOEBA_URL, method="HEAD")).headers.get("Last-Modified")
            date = datetime.datetime.strptime(modified, "%a, %d %b %Y %H:%M:%S %Z").strftime("%Y-%m-%d")
        except Exception:
            date = datetime.date.today().isoformat()
        TATOEBA_INFO = {"sha256": z.sha256_file(TATOEBA_PATH), "date": date}
        print("Tatoeba の書き出し:", TATOEBA_INFO, "（控えの場所:", TATOEBA_PATH + "）")
LLAMA = f"{WORK}/llama.cpp"
if not os.path.isdir(f"{LLAMA}/.git"):
    os.makedirs(LLAMA, exist_ok=True)
    subprocess.run(["git", "init", "-q", LLAMA], check=True)
    subprocess.run(["git", "-C", LLAMA, "remote", "add", "origin", LLAMA_URL], check=True)
subprocess.run(["git", "-C", LLAMA, "fetch", "-q", "--depth", "1", "origin", LLAMA_COMMIT], check=True)
subprocess.run(["git", "-C", LLAMA, "checkout", "-q", "FETCH_HEAD"], check=True)
print("そろいました。llama.cpp の版:",
      subprocess.run(["git", "-C", LLAMA, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())

## ⑥ 元の模型を学習できる形に戻す
zenz-v3.2-small は量子化（圧縮）された GGUF でしか配られていないため、圧縮をほどいて学習できる形にします（1分）。

In [ ]:
#@title ⑥ 元の模型を学習できる形に戻す
import os, json
BASE_DIR = f"{RUN_DIR}/base"
if not os.path.exists(f"{BASE_DIR}/from_gguf.json"):
    info = z.gguf_to_hf(BASE_GGUF, TOKENIZER_DIR, BASE_DIR, LLAMA)
else:
    info = json.load(open(f"{BASE_DIR}/from_gguf.json"))
print("戻しました:", info["config"], "埋め込みの出どころ:", info["embedding_from"])

## ⑦ 練習問題を作る
①の `MIX` の割合で、ウィキペディア（同音異義語・英単語まじりを含む）・RealPersonaChat の雑談・Tatoeba の話し言葉・
分野の文書・自作の一覧から問題を作ります。部品（text と wiki…）ごとに作ってドライブ（または手元）に残すので、
切れても足りない部品だけ作り直します。ウィキペディアは CPU 2 つで並べて作ります。
**試験の文（tests/conversion/ の全ファイル）と同じ文・それを含む文は、全部の出典で自動で捨てます。**
出典ごとの予定と実際の問題数、足りなかった分（ウィキペディアで埋めた数）が最後に表示されます。（数十分〜数時間。GPU は使いません）

In [ ]:
#@title ⑦ 練習問題を作る
import os, json, shlex, subprocess, sys, time, glob
os.makedirs(DATA_DIR, exist_ok=True)

def sh(*cmd):
    print("$", " ".join(shlex.quote(str(c)) for c in cmd), flush=True)
    subprocess.run([str(c) for c in cmd], check=True)

def last_line(path):
    try:
        with open(path, encoding="utf-8", errors="replace") as f:
            lines = [l.strip() for l in f.readlines()[-3:] if l.strip()]
        return lines[-1][:200] if lines else ""
    except OSError:
        return ""

BUILD = f"{KIT}/build_dataset.py"
if z.data_ready(DATA_DIR):
    print("練習問題は作り済みです:", DATA_DIR)
else:
    # 部品ごとに作る。作り終えた部品は DATA_DIR に残るので、切れても、①から押し直せば足りない部品だけ作る
    # （部品: text＝自作の一覧・分野・Tatoeba・RealPersonaChat、wiki…＝ウィキペディアを工程の数に分けたもの）。
    # 試験の文（tests/conversion/ の全ファイル）との重なりの除外は、全部の出典に効く
    for old in glob.glob(f"{DATA_DIR}/part-wiki*.json"):
        if os.path.basename(old)[5:-5] not in WIKI_PART_NAMES:
            raise SystemExit(f"{old} は、今の BUILD_PROCESSES = {BUILD_PROCESSES} と違う分け方で作った部品です。"
                             "BUILD_PROCESSES を戻すか、DATA_DIR の part-wiki* と train-wiki*・valid-wiki*・excluded-wiki* を消してください。")
    common = ["--out", DATA_DIR, "--repo", REPO, "--romaji-table", ROMAJI_TABLE, "--size", DATA_SIZE, "--seed", SEED,
              "--mix", MIX_ARG, "--max-passes", PASSES_ARG, "--kind-weights", KIND_WEIGHTS_ARG,
              "--ortho-share", ORTHO_SHARE, "--progress", 5000]
    SENTENCES = f"{WORK}/text/all.jsonl"
    # ---- 1. 文の一覧からの出典（text の部品）
    if TEXT_SOURCES_USED and not z.part_done(DATA_DIR, "text"):
        os.makedirs(f"{WORK}/text", exist_ok=True)
        config = json.load(open(f"{KIT}/domain_sources.json", encoding="utf-8"))
        git_sources = [s["name"] for s in config["sources"] if s.get("kind") not in ("external", "dialogue")]
        pieces = []
        if MIX.get("domain", 0) > 0:     # 分野の文書（版を固定して git で取り出す）。CPU で数分〜十数分
            sh(sys.executable, f"{KIT}/fetch_domain_sources.py", "--out", f"{WORK}/domain-src", "--only", *git_sources)
            sh(sys.executable, f"{KIT}/domain_text.py", "--src-root", f"{WORK}/domain-src", "--out", f"{WORK}/text",
               "--seed", 20261015, "--only", *git_sources)
            pieces.append(f"{WORK}/text/sentences.jsonl")
        if MIX.get("persona", 0) > 0:    # RealPersonaChat（版を固定）。発話の文字だけを使う
            sh(sys.executable, f"{KIT}/fetch_domain_sources.py", "--out", f"{WORK}/domain-src", "--only", "persona")
            sh(sys.executable, f"{KIT}/persona_text.py", "--src-root", f"{WORK}/domain-src", "--out", f"{WORK}/text")
            pieces.append(f"{WORK}/text/persona-train.jsonl")
        if MIX.get("tatoeba", 0) > 0:    # Tatoeba の話し言葉（試験の枠は文の番号で取り分けてある）
            sh(sys.executable, f"{REPO}/tools/colloquial_text.py", "--tatoeba", TATOEBA_PATH, "--table", ROMAJI_TABLE,
               "--out-dir", f"{WORK}/tatoeba", "--repo", REPO, "--dump-sha256", TATOEBA_INFO["sha256"],
               "--dump-date", TATOEBA_INFO["date"])
            subprocess.run(["cp", f"{WORK}/tatoeba/colloquial-counts.json", f"{WORK}/text/"], check=True)
            pieces.append(f"{WORK}/tatoeba/colloquial-train.jsonl")
        with open(SENTENCES, "wb") as out:
            for p in pieces:
                out.write(open(p, "rb").read())
        sh(sys.executable, BUILD, *common, "--domain-text", SENTENCES, "--only-source", *TEXT_SOURCES_USED, "--part", "text")
    # ---- 2. ウィキペディア（text の足りなかった分を埋める。工程を並べて動かす）
    todo = [(i, n) for i, n in enumerate(WIKI_PART_NAMES) if not z.part_done(DATA_DIR, n)]
    if todo:
        if not WIKI_PATHS:
            raise SystemExit("ウィキペディアのファイルがありません。⑤をもう一度押してください。")
        procs = []
        for i, name in todo:
            log = f"{WORK}/build-{name}.log"
            cmd = [sys.executable, BUILD, *map(str, common), "--english-only-exact", "--wiki", *WIKI_PATHS,
                   "--only-source", "wiki", "--slice", f"{i}/{WIKI_PROCS}", "--part", name]
            print("$", " ".join(shlex.quote(c) for c in cmd[:3]), "…", name, flush=True)
            procs.append((name, log, subprocess.Popen(cmd, stdout=open(log, "w"), stderr=subprocess.STDOUT)))
        while any(p.poll() is None for _, _, p in procs):
            time.sleep(120)
            for name, log, p in procs:
                print(f"  [{name}] {last_line(log)}", flush=True)
        for name, log, p in procs:
            if p.returncode != 0:
                print(open(log, encoding="utf-8", errors="replace").read()[-3000:])
                raise SystemExit(f"{name} の問題作りが失敗しました（上の表示）。もう一度⑦を押すと、足りない部品だけ作り直します。")
            print(f"  [{name}] {last_line(log)}")
    # ---- 3. 集める（valid.jsonl と manifest.json）
    sh(sys.executable, BUILD, "--out", DATA_DIR, "--repo", REPO, "--merge-parts")
    if not z.data_ready(DATA_DIR):
        raise SystemExit("問題作りが失敗しました。上の表示を確かめてください。")
manifest = json.load(open(f"{DATA_DIR}/manifest.json", encoding="utf-8"))
print("学習用:", manifest["counts"]["train"], "問 ／ 確かめ用:", manifest["counts"]["valid"], "問")
for source, row in manifest.get("mix_result", {}).items():
    print(f"  {source:<10} 予定 {row['share_planned']:6.1%}（{row['train_planned']:>9,} 問）→ 実際 {row['share_actual']:6.1%}"
          f"（{row['train']:>9,} 問）  足りない {row['short']:,}")
print("種類ごと:", manifest["counts"]["train_by_kind"])
print("試験と重なって捨てた文:", manifest["test_exclusion"]["excluded_sentences"])
if ONLY_DATA:
    print("\nONLY_DATA = True なので、ここで終わりです。ランタイムを GPU（T4 など）に替え、①の ONLY_DATA を False にして、"
          "①から押し直すと、作った練習問題を使って学習に進みます。")
    raise SystemExit

## ⑧ 元の模型を確かめる
文の終わりの記号を元の模型に合わせ、学習前の成績を測っておきます（5分）。

In [ ]:
#@title ⑧ 元の模型を確かめる
import os, json, numpy as np, torch
from transformers import AutoTokenizer, GPT2LMHeadModel, GPT2Tokenizer
DEVICE = torch.device("cuda")
tok = AutoTokenizer.from_pretrained(BASE_DIR)
slow = GPT2Tokenizer.from_pretrained(BASE_DIR)
sample = [json.loads(l) for _, l in zip(range(300), open(f"{DATA_DIR}/valid.jsonl", encoding="utf-8"))]
for r in sample:
    text = r["prompt"] + r["output"]
    assert tok(text, add_special_tokens=False)["input_ids"] == slow(text, add_special_tokens=False)["input_ids"], text
print("分かち方の部品: 速い版と遅い版の結果が同じです")

base = GPT2LMHeadModel.from_pretrained(BASE_DIR).to(DEVICE).eval()
END_FILE = f"{RUN_DIR}/end_token.json"
if not os.path.exists(END_FILE):
    probe, _ = z.load_examples(f"{DATA_DIR}/valid.jsonl", tok, end_id=3)
    choice = z.choose_end_token(base, probe, DEVICE, PROFILE["precision"])
    json.dump(choice, open(END_FILE, "w"))
END_ID = json.load(open(END_FILE))["end_id"]
print("文の終わりの記号:", END_ID, tok.convert_ids_to_tokens(END_ID), json.load(open(END_FILE))["votes"])

# 練習問題を数字にする（部品のまま読む。数百万問だと数分かかる。できた物は NPZ_DIR に控える）
TRAIN_FILES = [f"{DATA_DIR}/{f['name']}" for f in manifest.get("train_files") or [{"name": "train.jsonl"}]]
def cached(name, files):
    path = f"{NPZ_DIR}/{name}-{END_ID}.npz"
    if not os.path.exists(path):
        ex, dropped = z.load_examples(files, tok, END_ID)
        ex.save(path)
        print(f"{name}: {len(ex):,} 問（長すぎて外した {dropped} 問）→ {path}（{os.path.getsize(path) / 1e6:.0f}MB）")
    return z.Examples.load(path)
TRAIN = cached("train", TRAIN_FILES); VALID = cached("valid", [f"{DATA_DIR}/valid.jsonl"])
if len(TRAIN) < manifest["counts"]["train"] * 0.999:
    print(f"注意: 目録では {manifest['counts']['train']:,} 問のはずが {len(TRAIN):,} 問です。数字にした控え（npz）が古いかもしれません")
avg = float(TRAIN.length.mean())
est = z.estimate(PROFILE, len(TRAIN), EPOCHS, avg_tokens=avg, data_built=True, run_minutes=RUN_MINUTES or 170)
print(f"1問の平均 {avg:.0f} 字。学習の見積り 約{est['train_minutes'] / 60:.1f} 時間（約 {est['runs']} 回の接続に分かれます）")

BASE_EVAL = f"{RUN_DIR}/eval_base.json"
if not os.path.exists(BASE_EVAL):
    report = z.evaluate(base, VALID, tok, END_ID, DEVICE, PROFILE["precision"], per_kind=EVAL_PER_KIND, seed=SEED)
    json.dump(report, open(BASE_EVAL, "w"), ensure_ascii=False, indent=1)
z.print_report(json.load(open(BASE_EVAL)))
for kind, r in json.load(open(BASE_EVAL)).items():
    for e in r["examples"][:1]:
        print(kind, "| 正解:", e["answer"], "| 元の模型:", e["got"])
del base; torch.cuda.empty_cache()

## ⑨ 学習する（いちばん長い）
進み具合と残り時間・残りのユニットを表示します。**途中で切れても、①から押し直せば続きから始まります。**
最新の途中の結果だけをドライブに残します（約1.1GB）。①の `RUN_MINUTES` に上限（分）を入れると、その時間で止まって保存します。
止まったら⑨をもう一度押してください（手元の小さな模型で、26回に分けても切れずに学習したときと完全に同じ結果になることを確かめてあります）。

In [ ]:
#@title ⑨ 学習する
import os, shutil
FINAL_DIR = f"{RUN_DIR}/final"
CKPT_DIR = f"{RUN_DIR}/checkpoints"
if os.path.exists(f"{FINAL_DIR}/train_state.json"):
    print("学習はもう終わっています:", FINAL_DIR)
else:
    # 最新の途中の結果から続ける（最新の KEEP_CHECKPOINTS 個だけドライブに残す）。RUN_MINUTES の上限で止まったときは None が返る
    done = z.train(BASE_DIR, TRAIN, VALID, CKPT_DIR, FINAL_DIR, epochs=EPOCHS, lr=LEARNING_RATE,
                   profile=PROFILE, seed=SEED, checkpoint_minutes=CHECKPOINT_MINUTES, keep=KEEP_CHECKPOINTS,
                   stop_after_seconds=(RUN_MINUTES * 60 if RUN_MINUTES else None))
    if done is None:
        raise SystemExit("今回の分はここまでです。⑨をもう一度押す（接続が切れていたら①から押し直す）と、続きから始まります。")
if CLEAN_CHECKPOINTS and os.path.exists(f"{FINAL_DIR}/train_state.json") and os.path.isdir(CKPT_DIR):
    shutil.rmtree(CKPT_DIR)
    print("途中の結果を消しました（約1.1GB の空き）")

## ⑩ 学習の前と後の成績を比べる
同じ確かめ用の問題（学習には使っていない）で比べます。「差」がマイナスの種類があれば、表示の注意を読んでください。

In [ ]:
#@title ⑩ 成績を比べる
import json, torch
from transformers import GPT2LMHeadModel
if not os.path.exists(f"{FINAL_DIR}/train_state.json"):
    raise SystemExit("学習がまだ終わっていません。⑨を押して、続きから学習してください。")
TUNED_EVAL = f"{RUN_DIR}/eval_tuned.json"
if not os.path.exists(TUNED_EVAL):
    tuned = GPT2LMHeadModel.from_pretrained(FINAL_DIR).to(DEVICE).eval()
    report = z.evaluate(tuned, VALID, tok, END_ID, DEVICE, PROFILE["precision"], per_kind=EVAL_PER_KIND, seed=SEED)
    json.dump(report, open(TUNED_EVAL, "w"), ensure_ascii=False, indent=1)
    del tuned; torch.cuda.empty_cache()
before, after = json.load(open(BASE_EVAL)), json.load(open(TUNED_EVAL))
z.print_report(before, after)
worse = [k for k in ("wiki_whole", "wiki_context", "wiki_span") if k in after and after[k]["exact"] < before[k]["exact"] - 0.01]
if worse:
    print("\n注意: ふつうの文の成績が下がっています:", worse,
          "\nこの模型は試験で悪くなる見込みが高いです。①の MIX で wiki の割合を増やす（雑談・分野を減らす）か EPOCHS を 1 にし、"
          "RUN_NAME を変えて作り直すことを考えてください。")
else:
    print("\nふつうの文の成績は下がっていません。")

## ⑪ GGUF にして縮める
llama.cpp（コンパウンドIME と同じ版）で GGUF にし、元と同じ Q5_K_M に量子化します。
比べるための「学習していない元の模型を同じ手順で作り直したもの」（対照）も作ります（10分）。

In [ ]:
#@title ⑪ GGUF にして縮める
import os, subprocess, sys
BIN = f"{LLAMA}/build/bin"
if not os.path.exists(f"{BIN}/llama-quantize"):
    subprocess.run(["cmake", "-S", LLAMA, "-B", f"{LLAMA}/build", "-DGGML_CUDA=OFF", "-DLLAMA_CURL=OFF",
                    "-DCMAKE_BUILD_TYPE=Release"], check=True, capture_output=True)
    subprocess.run(["cmake", "--build", f"{LLAMA}/build", "-j", str(os.cpu_count() or 2), "--target",
                    "llama-quantize", "llama-tokenize", "llama-simple"], check=True, capture_output=True)
OUT = f"{RUN_DIR}/result"
os.makedirs(OUT, exist_ok=True)
NAME = f"zenz-v3.2-small-ft-{RUN_NAME}"
TARGETS = {"tuned": (FINAL_DIR, f"{OUT}/{NAME}-Q5_K_M.gguf"),
           "control": (BASE_DIR, f"{OUT}/zenz-v3.2-small-roundtrip-Q5_K_M.gguf")}
for label, (src, dst) in TARGETS.items():
    if os.path.exists(dst):
        continue
    z.prepare_for_convert(src)
    f32 = f"{WORK}/{label}-f32.gguf"
    subprocess.run([sys.executable, f"{LLAMA}/convert_hf_to_gguf.py", src, "--outtype", "f32", "--outfile", f32,
                    "--model-name", NAME if label == "tuned" else "zenz-v3.2-small-roundtrip"], check=True)
    subprocess.run([f"{BIN}/llama-quantize", f32, dst, "Q5_K_M"], check=True, capture_output=True)
    print("作りました:", dst)

## ⑫ できた GGUF を確かめる
元の GGUF と設定・重みの形・量子化の種類が同じか、分かち方が同じか、llama.cpp で動かした答えが学習した模型の答えと合うかを調べます。

In [ ]:
#@title ⑫ できた GGUF を確かめる
import json, torch
from transformers import GPT2LMHeadModel
original = z.gguf_summary(BASE_GGUF, LLAMA)
ok = True
for label, (_, path) in TARGETS.items():
    problems = z.compare_summaries(original, z.gguf_summary(path, LLAMA))
    print(f"{label}: 元の GGUF との違い {len(problems)} 件")
    for p in problems[:10]:
        print("   ", p)
    ok &= not problems
records = [json.loads(l) for _, l in zip(range(40), open(f"{DATA_DIR}/valid.jsonl", encoding="utf-8"))]
same_tok = 0
for r in records[:20]:
    text = r["prompt"] + r["output"]
    same_tok += z.llama_tokenize_ids(f"{BIN}/llama-tokenize", TARGETS["tuned"][1], text) == \
        tok(text, add_special_tokens=False)["input_ids"]
print(f"分かち方が同じ: {same_tok}/20")
ok &= same_tok == 20
tuned = GPT2LMHeadModel.from_pretrained(FINAL_DIR).to(DEVICE).eval()
prompts = [tok(r["prompt"], add_special_tokens=False)["input_ids"] for r in records]
hf = [tok.decode(x) for x in z.greedy(tuned, prompts, {END_ID}, DEVICE, "fp32")]
agree = 0
for r, h in zip(records, hf):
    g = z.llama_simple_answer(f"{BIN}/llama-simple", TARGETS["tuned"][1], r["prompt"])
    agree += g == h
print(f"llama.cpp（量子化後）と学習した模型の答えが同じ: {agree}/{len(records)}（量子化のため全部は合わないことがあります）")
del tuned; torch.cuda.empty_cache()
print("確認: よし" if ok and agree >= len(records) * 0.8 else "確認: 気になる点があります。上の表示を見てください。")

## ⑬ 結果をドライブに保存する
SHA256（指紋）、説明書き（出典と利用条件）、問題の目録、成績を `result` フォルダにまとめます。

In [ ]:
#@title ⑬ 結果をまとめる
import json, shutil
lines = []
for label, (_, path) in TARGETS.items():
    digest = z.sha256_file(path)
    lines.append(f"{digest}  {os.path.basename(path)}")
    print(label, os.path.basename(path), os.path.getsize(path) // 1_000_000, "MB", digest)
open(f"{OUT}/SHA256SUMS.txt", "w").write("\n".join(lines) + "\n")
tuned_path = TARGETS["tuned"][1]
open(f"{OUT}/MODEL_CARD.md", "w").write(z.MODEL_CARD.format(name=os.path.basename(tuned_path), base_url=BASE_GGUF_URL,
    epochs=EPOCHS, lr=LEARNING_RATE, sha256=z.sha256_file(tuned_path), materials=z.materials_text(manifest)))
shutil.copy(f"{DATA_DIR}/manifest.json", f"{OUT}/manifest.json")
json.dump({"base": json.load(open(BASE_EVAL)), "tuned": json.load(open(TUNED_EVAL)), "repo_commit": REPO_COMMIT,
           "llama_commit": LLAMA_COMMIT, "end_id": END_ID, "profile": PROFILE, "mix": MIX, "kind_weights": KIND_WEIGHTS, "ortho_share": ORTHO_SHARE, "max_passes": MAX_PASSES,
           "data_size": DATA_SIZE, "tatoeba": TATOEBA_INFO},
          open(f"{OUT}/eval.json", "w"), ensure_ascii=False, indent=1)
print("\nできあがり。Google ドライブの次の場所にあります:", OUT)
print("※ この模型はまだ公開しないでください。比べ方は README.md を見てください。")